In [35]:
!pip install datasets
from datasets import load_dataset
import pandas as pd
import re


Step 1:- Install and load dataset

In [54]:
from datasets import load_dataset

dataset = load_dataset("wangrongsheng/ag_news")
df = pd.DataFrame(dataset['train'])
print(df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [38]:
# Display the first training example
print(dataset["train"][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


Step 2: Clean and Normalize the Text

In [39]:
# Import regular expressions
import re

# Function to clean text
def clean_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-z\s]', '', text)

    return text

In [40]:
# Clean all training articles
train_text = [clean_text(article["text"]) for article in dataset["train"]]

# Clean all testing articles
test_text = [clean_text(article["text"]) for article in dataset["test"]]

# Store labels
train_labels = [article["label"] for article in dataset["train"]]
test_labels = [article["label"] for article in dataset["test"]]

print(train_text[0])
print(train_labels[0])

wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again
2


Step 3: Tokenize the Strings into Numbers

In [41]:
# Import Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer

In [42]:
# Create tokenizer

tokenizer = Tokenizer(num_words=10000)

# Learn vocabulary from training data
tokenizer.fit_on_texts(train_text)

# Convert text into sequences of integers
X_train = tokenizer.texts_to_sequences(train_text)
X_test = tokenizer.texts_to_sequences(test_text)

In [43]:
# Display one converted sequence

print(X_train[0])

[391, 324, 1525, 99, 54, 1, 812, 23, 23, 391, 1988, 4, 34, 3893, 737, 295]


Step 4: Pad and Truncate Sequences

In [44]:
# Import pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [45]:
# Maximum number of words

max_length = 50

# Pad training data
X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# Pad testing data
X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print(X_train.shape)

(120000, 50)


Step 5: Convert Labels to Categorical Vectors

In [46]:
# Import one-hot encoder
from tensorflow.keras.utils import to_categorical

In [47]:
# Convert labels

y_train = to_categorical(train_labels)

y_test = to_categorical(test_labels)

print(y_train[0])

[0. 0. 1. 0.]


Step 6: Define the RNN Model

In [48]:
# Import required layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dense

In [49]:
# Create Sequential model

model = Sequential()

# Embedding layer
model.add(
    Embedding(
        input_dim=10000,
        output_dim=64,
        input_length=max_length
    )
)

# Simple RNN layer
model.add(SimpleRNN(64))

# Output layer
model.add(Dense(4, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [50]:
# Display model summary
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Step 7: Compile and Train the Model

In [51]:
# Compile the model

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [52]:
# Train the model

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.8394 - loss: 0.4841 - val_accuracy: 0.8764 - val_loss: 0.3844
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 23s 20ms/step - accuracy: 0.9079 - loss: 0.2951 - val_accuracy: 0.8774 - val_loss: 0.3857
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.9198 - loss: 0.2557 - val_accuracy: 0.8804 - val_loss: 0.3711
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.9303 - loss: 0.2214 - val_accuracy: 0.8855 - val_loss: 0.3642
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.9373 - loss: 0.1968 - val_accuracy: 0.8714 - val_loss: 0.4117


Step 8: Evaluate the Model

In [53]:
# Evaluate the model

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8847 - loss: 0.3647
Test Loss : 0.3646669387817383
Test Accuracy : 0.8847368359565735
